### **Student Information**
GitHub ID: JasonSu0205

Kaggle name: Jason_Su_0205

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

***

# **Project Report**

---

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

* **讀取與合併**：`final_posts.json`（貼文內容）、`data_identification.csv`（train/test 劃分）、`emotion.csv`（train 標籤）以 `post_id` 合併。train 3,360 則、test 1,440 則，6 種情緒（joy 875、anger 674、sadness 628、disgust 454、surprise 441、fear 288），沒有空白貼文或重複 id。
* **清理**：網址換成 `URL`、`@帳號` 換成 `USER`、HTML 跳脫字元還原、多餘空白合併，避免和情緒無關的雜訊干擾模型。
* **保留大小寫和標點**：像 `NO!!!` 的驚嘆號與全大寫本身就帶有情緒強度線索，所以不轉小寫也不刪標點（TF-IDF 內部再自行轉小寫）。

### 1.2 Feature Engineering Steps

五種特徵接在一起使用：

| 特徵 | 維度 | 作用 |
|---|---|---|
| TF-IDF（字詞 1–2 gram + 字元 2–5 gram） | 約 82,000 | 抓出明顯的情緒關鍵字；字元 n-gram 能接住拼錯字和變形字 |
| MiniLM 句向量（all-MiniLM-L6-v2） | 384 | 理解整句的語意，不只看字面 |
| 預訓練情緒模型機率（j-hartmann/emotion-english-distilroberta-base） | 7 | 借用已在大量情緒資料上訓練過的模型經驗 |
| GoEmotions 機率（SamLowe/roberta-base-go_emotions） | 28 | 以 Reddit 留言訓練，貼近本資料的語境，能補出更細的情緒分布 |
| bge-large 句向量（BAAI/bge-large-en-v1.5） | 1,024 | 比 MiniLM 大很多的句向量模型，語意表示更細 |

稠密特徵（後四種）合併時乘上權重，避免被維度很大的稀疏 TF-IDF 淹沒；權重用 5-fold 交叉驗證挑選。

### 1.3 Explanation of Your Model

* **基礎模型**：所有特徵接起來交給 Linear SVM（`C=0.1`, `class_weight="balanced"`）打底。
* **驗證方式**：test 沒有標籤，所以用 train 的 5-fold 交叉驗證（macro F1）比較每個版本，確認有進步才上傳 Kaggle。
* **進步過程**：

| 版本 | 5-fold macro F1 | Kaggle Public |
|---|---|---|
| A. 只用 TF-IDF + Linear SVM | 0.513 | – |
| D. TF-IDF + MiniLM + 情緒模型 + Linear SVM | 0.609 | 0.5899 |
| D + 微調 distilroberta 情緒模型（混合） | 驗證集 0.635 | 0.6092 |
| D2 = D + GoEmotions | 0.629（驗證集 0.666） | 0.6173 |
| D2 + 5-fold 微調 roberta-large（混合，比重 0.2） | 0.649 | 0.6296 |
| 再加 5-fold 微調 GoEmotions roberta-base（三模型混合 0.8/0.15/0.05） | 0.652 | 0.6330 |
| SVM 特徵再加 bge-large（三模型混合 0.75/0.15/0.1） | 0.655 | 0.6347 |
| 再加 Gemini zero-shot（四模型混合 0.65/0.15/0.1/0.1）← 最終版本 | 0.659 | **0.638** |

* **最終模型**：四個模型加權平均——Linear SVM（全部五種特徵）、5-fold 微調 roberta-large、5-fold 微調 GoEmotions roberta-base、Gemini（gemini-3.1-flash-lite）zero-shot。表格前面幾版是實驗過程，notebook 只保留最終版本的程式碼。
* **微調（fine-tuning）**：用 train 資料繼續訓練預訓練情緒模型本身。它原本的類別已包含本比賽的 6 種情緒，所以保留原分類層、把標籤對應過去。roberta-large 版本用 RTX 5060（bf16）做 5-fold，test 預測取 5 個模型平均。
* **混合（ensemble）**：SVM 的分數（softmax 後）和微調模型的機率加權平均，比重用驗證資料挑。不同模型犯的錯不同，加權平均能互補。
* **過擬合的檢查**：每折挑最好的 epoch、在 OOF 上挑混合比重，都會讓 OOF 分數略為高估（OOF 0.65 vs Public 0.63），所以 0.002–0.003 的差距視為雜訊。OOF 和 Public 的落差一直維持在約 0.02，沒有越拉越大，代表後面的改進沒有更嚴重的過擬合，只是幅度小。最終勾選 Gemini 四模型版（0.638）和較少調參的 roberta-large 雙模型版（0.6296），兩個結構不同，私人排行榜取較高者。結果 Private 排行榜分數 0.6576、第 14 名（比 Public 名次進步 15 名），可見沒有對 Public 過擬合。
* **Gemini zero-shot**：不訓練，直接請 Gemini 對每則貼文給 6 種情緒的分數（每次送 20 則，回傳 JSON）。單獨只有 0.559，但它判斷的方式和其他模型完全不同，混合後 OOF 0.655 → 0.659、Public 0.6347 → 0.638。因為它沒看過 train 標籤，在 train 上的分數不會被高估。
* **試過但沒採用**：DeBERTa-v3-large（新接 6 類分類層）。一開始權重以 float16 載入導致訓練數值爆掉；改成 float32 後雖然能學，但前兩輪驗證 F1 只有 0.17、0.37，每輪約 10 分鐘，跑完 5 折要 2.5 小時，預期效益太低所以放棄。另外試過用 LogisticRegression 做 stacking，分數沒有比加權平均好。也試過在提示裡附上 train 範例的 Gemini few-shot 版，但跑到一半免費 API 額度用完，test 沒能做完，所以沒採用。
* **最有用的一步**：加入外部預訓練模型的情緒機率（情緒模型 + GoEmotions），讓分數從 0.51 拉到 0.63；單純換分類器或調參數的幫助小很多。

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

我使用 Claude（AI 助教）協助這次比賽，程式碼由 Claude 撰寫、我在自己電腦上執行並回報結果。我給 AI 的主要 prompt：

1. 「今天完成（Kaggle）」、「你自己去頁面看」：請 AI 讀比賽頁面與資料格式。
2. 「能不能表現再好一點」、「再往上提升」：請 AI 提出提升分數的方法。
3. 「我有 PC 可以盡量操」：告知有 RTX 5060，請 AI 改用 GPU 訓練更大的模型。
4. 「現在做的都有符合作業規定嗎」：請 AI 對照比賽規則與作業說明檢查。
5. 「c6 幫我註解改一下，簡短的」：請 AI 把程式註解改短。
6. 「解釋一下這段的語法」：請 AI 解釋整個流程。
7. 「會不會有過擬合的問題，你這樣搞」：請 AI 檢查挑 epoch 與混合比重是否讓驗證分數高估。
8. 「說不定 train 的比較差，反而比較好吧」：質疑只看 OOF 分數挑模型，最後決定兩個版本都上傳並都勾選。
9. 「有沒有更強的模型可以用」、「直接做最強的模型」：請 AI 嘗試更大的模型（bge-large、DeBERTa-v3-large）。
10. 「我們在用新模型的時候前面的程式碼不用改嗎，感覺這樣有點屎山代碼了」：請 AI 把一路疊加的實驗程式碼整理成精簡的最終版本。
11. 「0.6347 根本沒有很大進步，我覺得是有點 overfitting 了」、「那你覺得 0.67 是怎麼辦到的」：討論還能怎麼進步，AI 推測前段班可能用了大型語言模型。
12. 「好，試試看第一種」：決定用 Gemini 當第四個模型。
13. 「能不能大改方案，用更強的算力」：討論改用 Kaggle GPU 微調 7B 語言模型，評估截止時間和風險後，先試 Gemini few-shot；額度用完後決定以 0.638 的版本作為最終提交。

我採用的建議與原因：

* **保留大小寫與標點**：驚嘆號和全大寫帶有情緒強度，刪掉會損失資訊。
* **加入外部情緒模型與 GoEmotions 的機率特徵**：單靠 TF-IDF 只有 0.51，加入後到 0.63，能補齊單一字詞特徵的盲點。
* **用 5-fold 交叉驗證比較版本**：每天只能上傳 5 次，而且 test 沒有標籤，交叉驗證能在上傳前客觀確認改進是否真的有效。
* **微調與模型混合**：不同模型犯的錯不同，混合能互補；有 GPU 後改用較大的 roberta-large。
* **加入 bge-large、放棄 DeBERTa**：bge 只多花幾分鐘、OOF 有小幅進步就採用；DeBERTa 成本太高、前兩輪分數太低就停掉。
* **加入 Gemini**：大型語言模型對語氣、諷刺的理解和 roberta 不同，錯的題目不一樣，混合能互補；結果存檔，重跑 notebook 不會再呼叫 API。
* **整理程式碼**：兩個微調模型共用同一個函式，所有特徵與微調結果存檔，重跑幾分鐘就能重現最終預測。
* **提交格式修正**：sampleSubmission 的欄位是 `post_id`，但 Kaggle 要求 `id`，第一次上傳失敗後依 AI 建議改正。

---


---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
### Add the code related to the preprocessing steps in cells inside this section
import json, re
import numpy as np
import pandas as pd
 
DATA_DIR = "data/kaggle"
SEED = 42
 
posts = pd.DataFrame(json.load(open(f"{DATA_DIR}/final_posts.json", encoding="utf-8")))
split = pd.read_csv(f"{DATA_DIR}/data_identification.csv")
labels = pd.read_csv(f"{DATA_DIR}/emotion.csv")
df = posts.merge(split, on="post_id", how="inner")
 
# 輕度清理：網址、@帳號換成代號，大小寫和標點保留（帶情緒）
def clean(text):
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", str(text))
    text = re.sub(r"@\w+", " USER ", text)
    text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
    return re.sub(r"\s+", " ", text).strip()
 
df["clean_text"] = df["text"].apply(clean)
train_df = df[df["set"] == "train"].merge(labels, on="post_id", how="inner").reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)
 
print("train:", train_df.shape, " test:", test_df.shape)
print(train_df["label"].value_counts())
 

train: (3360, 5)  test: (1440, 4)
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64


## 2. Feature Engineering Steps

In [2]:
### Add the code related to the feature engineering steps in cells inside this section
import os, gc
import torch
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from transformers import pipeline
 
FEAT_DIR = "data/kaggle/features"
os.makedirs(FEAT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
all_text = pd.concat([train_df["clean_text"], test_df["clean_text"]]).tolist()
n_train = len(train_df)
 
# 算過的特徵存成 .npy，重跑直接讀
def cached(name, fn):
    path = f"{FEAT_DIR}/{name}.npy"
    if not os.path.exists(path):
        np.save(path, fn())
        gc.collect(); torch.cuda.empty_cache()
    return np.load(path)
 
def sbert(model_name):
    return lambda: SentenceTransformer(model_name, device=device).encode(
        all_text, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
 
def emotion_probs(model_name):
    def run():
        clf = pipeline("text-classification", model=model_name, top_k=None, truncation=True, device=device)
        out = clf(all_text, batch_size=32)
        names = sorted(d["label"] for d in out[0])
        return np.array([[{d["label"]: d["score"] for d in row}[n] for n in names] for row in out])
    return run
 
# 1. TF-IDF：字詞 1-2 gram + 字元 2-5 gram
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, lowercase=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)
X_tfidf = hstack([word_tfidf.fit_transform(all_text), char_tfidf.fit_transform(all_text)]).tocsr()
 
# 2. 句向量：MiniLM（小）和 bge-large（大）
X_minilm = cached("minilm", sbert("sentence-transformers/all-MiniLM-L6-v2"))
X_bge = cached("bge_large", sbert("BAAI/bge-large-en-v1.5"))
 
# 3. 現成情緒模型的輸出機率：7 類 + GoEmotions 28 類
X_emo = cached("emo_distilroberta", emotion_probs("j-hartmann/emotion-english-distilroberta-base"))
X_go = cached("go_emotions", emotion_probs("SamLowe/roberta-base-go_emotions"))
 
# 接成一個矩陣；權重用 5-fold CV 試過（見報告 1.3）
W = {"minilm": 1, "emo": 3, "go": 2, "bge": 2}
X = hstack([X_tfidf, csr_matrix(X_minilm * W["minilm"]), csr_matrix(X_emo * W["emo"]),
            csr_matrix(X_go * W["go"]), csr_matrix(X_bge * W["bge"])]).tocsr()
print("TF-IDF", X_tfidf.shape, "MiniLM", X_minilm.shape, "bge", X_bge.shape,
      "emo", X_emo.shape, "go", X_go.shape, "-> all", X.shape)
 

TF-IDF (4800, 81817) MiniLM (4800, 384) bge (4800, 1024) emo (4800, 7) go (4800, 28) -> all (4800, 83260)


## 3. Model Implementation Steps

In [3]:
### Add the code related to the model implementation steps in cells inside this section
from scipy.special import softmax
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
 
y = train_df["label"].values
classes = sorted(set(y))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
macro_f1 = lambda p: f1_score(y, np.array(classes)[p.argmax(1)], average="macro")
 
# 模型 1：LinearSVC。OOF = 每則 train 都由沒看過它的那一折預測
svm = LinearSVC(C=0.1, class_weight="balanced", max_iter=5000, random_state=SEED)
oof_svm = softmax(cross_val_predict(svm, X[:n_train], y, cv=cv, method="decision_function"), axis=1)
svm.fit(X[:n_train], y)
test_svm = softmax(svm.decision_function(X[n_train:]), axis=1)
print(f"LinearSVC 5-fold OOF macro F1: {macro_f1(oof_svm):.4f}")
 

LinearSVC 5-fold OOF macro F1: 0.6331


In [4]:
# 模型 2、3：用 GPU 微調兩個現成情緒模型（5-fold），每折存檔
# 兩個模型原本的類別都包含這 6 種情緒，所以沿用原分類層，只把標籤對過去
import time
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
 
EPOCHS, MAX_LEN = 3, 128
texts, test_texts = train_df["clean_text"].values, test_df["clean_text"].values
 
def finetune(model_name, tag, batch, accum, lr):
    tok = AutoTokenizer.from_pretrained(model_name)
 
    def loader(model, txts, labels=None, shuffle=False):
        lab = [model.config.label2id[l] for l in labels] if labels is not None else [-1] * len(txts)
        def collate(b):
            enc = tok([t for t, _ in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
            enc["labels"] = torch.tensor([l for _, l in b])
            return enc
        return DataLoader(list(zip(txts, lab)), batch_size=batch if labels is not None else batch * 2,
                          shuffle=shuffle, collate_fn=collate)
 
    def proba(model, txts, ids):
        model.eval(); out = []
        with torch.no_grad(), torch.autocast(device, dtype=torch.bfloat16, enabled=device == "cuda"):
            for b in loader(model, txts):
                b.pop("labels")
                logits = model(**{k: v.to(device) for k, v in b.items()}).logits[:, ids]
                out.append(torch.softmax(logits.float(), 1).cpu().numpy())
        return np.vstack(out)
 
    oof, test = np.zeros((n_train, len(classes))), np.zeros((len(test_df), len(classes)))
    for k, (tr, va) in enumerate(cv.split(texts, y)):
        path = f"{FEAT_DIR}/{tag}_fold{k}.npz"
        if not os.path.exists(path):
            torch.manual_seed(SEED + k)
            model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
            model.config.problem_type = "single_label_classification"
            ids = [model.config.label2id[c] for c in classes]
            dl = loader(model, texts[tr], y[tr], shuffle=True)
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
            steps = EPOCHS * len(dl) // accum
            sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
            best = -1
            for ep in range(1, EPOCHS + 1):
                model.train(); t0 = time.time()
                for i, b in enumerate(dl, 1):
                    with torch.autocast(device, dtype=torch.bfloat16, enabled=device == "cuda"):
                        loss = model(**{n: v.to(device) for n, v in b.items()}).loss / accum
                    loss.backward()
                    if i % accum == 0 or i == len(dl):
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        opt.step(); sch.step(); opt.zero_grad()
                va_p = proba(model, texts[va], ids)
                f1 = f1_score(y[va], np.array(classes)[va_p.argmax(1)], average="macro")
                print(f"  {tag} fold {k} epoch {ep}: macro F1 {f1:.4f} ({time.time() - t0:.0f}s)")
                if f1 > best:   # 留驗證分數最好的那一輪
                    best, best_va, best_te = f1, va_p, proba(model, test_texts, ids)
            np.savez(path, va=best_va, te=best_te)
            del model, opt; gc.collect(); torch.cuda.empty_cache()
        d = np.load(path)
        oof[va], test = d["va"], test + d["te"] / cv.get_n_splits()
    print(f"{tag} 5-fold OOF macro F1: {macro_f1(oof):.4f}")
    return oof, test
 
oof_big, test_big = finetune("j-hartmann/emotion-english-roberta-large", "big", batch=8, accum=2, lr=1e-5)
oof_go, test_go = finetune("SamLowe/roberta-base-go_emotions", "go", batch=16, accum=1, lr=2e-5)
 

big 5-fold OOF macro F1: 0.6342
go 5-fold OOF macro F1: 0.6285


In [5]:
# 三個模型加權平均，比重在 OOF 上用網格找（間隔 0.05，加總為 1）
best_w, best_f = None, -1
for a in np.arange(0, 1.001, 0.05):
    for b in np.arange(0, 1.001 - a, 0.05):
        f1 = macro_f1((1 - a - b) * oof_svm + a * oof_big + b * oof_go)
        if f1 > best_f:
            best_w, best_f = (round(1 - a - b, 2), round(a, 2), round(b, 2)), f1
print(f"weights (SVM, roberta-large, go-roberta) = {best_w} -> OOF macro F1 {best_f:.4f}")
 
final = best_w[0] * test_svm + best_w[1] * test_big + best_w[2] * test_go
sample = pd.read_csv(f"{DATA_DIR}/sampleSubmission.csv")
sub = pd.DataFrame({"id": test_df["post_id"], "label": np.array(classes)[final.argmax(1)]})
sub = sample[["post_id"]].rename(columns={"post_id": "id"}).merge(sub, on="id", how="left")   # Kaggle 要 id 欄位
assert sub["label"].notna().all() and len(sub) == len(sample)
sub.to_csv("submission_final.csv", index=False)
print(sub["label"].value_counts())
 

weights (SVM, roberta-large, go-roberta) = (np.float64(0.75), np.float64(0.15), np.float64(0.1)) -> OOF macro F1 0.6554
label
joy         381
anger       340
sadness     265
surprise    200
disgust     142
fear        112
Name: count, dtype: int64


In [11]:
# 模型 4：讓 Gemini（大型語言模型）直接判斷情緒，不訓練（zero-shot）
# 一次送 20 則，回傳每則 6 種情緒的分數；結果存檔，中斷後重跑會接著做
import json, time, requests
 
env = dict(l.strip().split("=", 1) for l in open("config/.env", encoding="utf-8")
           if "=" in l and not l.strip().startswith("#"))
API_KEY = next(v.strip().strip('"').strip("'") for k, v in env.items() if "GEMINI" in k.upper() or "GOOGLE" in k.upper())
LLM_MODEL = "gemini-3.1-flash-lite"
LLM_PATH = f"{FEAT_DIR}/gemini.json"
URL = f"https://generativelanguage.googleapis.com/v1beta/models/{LLM_MODEL}:generateContent"
 
PROMPT = """You are labeling the emotion of Reddit posts. Choose from exactly these 6 emotions:
anger, disgust, fear, joy, sadness, surprise.
For each post, give a score from 0 to 100 for every emotion (scores of one post sum to 100).
Return only a JSON list with one object per post, in the same order, like
[{"id": "...", "anger": 0, "disgust": 0, "fear": 0, "joy": 0, "sadness": 0, "surprise": 0}]
 
Posts:
"""
 
THINK = [{"thinkingBudget": 0}, {"thinkingLevel": "minimal"}, None]   # 盡量關掉「思考」，比較快
 
def ask(batch):
    for attempt in range(6):
        cfg = {"temperature": 0, "responseMimeType": "application/json"}
        if THINK[0]: cfg["thinkingConfig"] = THINK[0]
        body = {"contents": [{"parts": [{"text": PROMPT + json.dumps(batch, ensure_ascii=False)}]}], "generationConfig": cfg}
        try:
            r = requests.post(URL, headers={"x-goog-api-key": API_KEY}, json=body, timeout=300)
        except requests.exceptions.RequestException as e:   # 逾時或斷線，等一下再試
            print("  retry:", type(e).__name__); time.sleep(20); continue
        if r.status_code == 400 and "thinking" in r.text.lower() and len(THINK) > 1:
            THINK.pop(0); continue
        if r.status_code in (429, 500, 503, 504):            # 超過速率限制或伺服器忙
            print("  retry:", r.status_code); time.sleep(30 * min(attempt + 1, 4)); continue
        r.raise_for_status()
        try:
            out = json.loads(r.json()["candidates"][0]["content"]["parts"][0]["text"])
            got = {str(o["id"]): [float(o.get(c) or 0) for c in classes] for o in out if isinstance(o, dict) and "id" in o}
        except Exception:
            got = {}
        ids = {b["id"] for b in batch}
        got = {k: v for k, v in got.items() if k in ids and sum(v) > 0}
        missing = [b for b in batch if b["id"] not in got]
        if not missing:
            return got
        if len(batch) > 1 and got:              # 少了幾則：缺的那幾則一則一則重問
            for b in missing:
                got.update(ask([b]))
            return got
        time.sleep(5)                           # 格式不對就重問
    if len(batch) == 1:                         # 真的問不出來的那一則，6 類給一樣的分數
        print("  give up:", batch[0]["id"]); return {batch[0]["id"]: [1.0] * len(classes)}
    got = {}
    for b in batch:
        got.update(ask([b]))
    return got
 
llm = json.load(open(LLM_PATH)) if os.path.exists(LLM_PATH) else {}
items = [{"id": str(i), "text": t} for i, t in zip(pd.concat([train_df["post_id"], test_df["post_id"]]), all_text)]
todo = [it for it in items if it["id"] not in llm]
print(f"done {len(llm)}, to do {len(todo)}")
t0 = time.time()
for s in range(0, len(todo), 20):
    llm.update(ask(todo[s:s + 20]))
    json.dump(llm, open(LLM_PATH, "w"))
    if s % 200 == 0:
        print(f"  {len(llm)}/{len(items)} ({time.time() - t0:.0f}s)")
print("finished", len(llm))
 

done 2840, to do 1960
  2860/4800 (109s)
  retry: 429
  3060/4800 (176s)
  retry: 429
  3260/4800 (242s)
  3460/4800 (284s)
  3660/4800 (314s)
  retry: 429
  3860/4800 (380s)
  4060/4800 (412s)
  retry: 429
  4260/4800 (474s)
  4460/4800 (505s)
  retry: 429
  4660/4800 (565s)
finished 4800


In [12]:
# Gemini 的分數換成機率，先看它單獨在 train 上準不準，再當第四個模型一起混合
def llm_proba(ids):
    p = np.array([llm[str(i)] for i in ids]) + 1e-3
    return p / p.sum(1, keepdims=True)
 
oof_llm, test_llm = llm_proba(train_df["post_id"]), llm_proba(test_df["post_id"])
print(f"Gemini zero-shot macro F1 on train: {macro_f1(oof_llm):.4f}")
 
best_w4, best_f4 = None, -1
for a in np.arange(0, 1.001, 0.05):
    for b in np.arange(0, 1.001 - a, 0.05):
        for c in np.arange(0, 1.001 - a - b, 0.05):
            f1 = macro_f1((1 - a - b - c) * oof_svm + a * oof_big + b * oof_go + c * oof_llm)
            if f1 > best_f4:
                best_w4, best_f4 = (round(1 - a - b - c, 2), round(a, 2), round(b, 2), round(c, 2)), f1
print(f"weights (SVM, roberta-large, go-roberta, Gemini) = {best_w4} -> OOF macro F1 {best_f4:.4f}   [3 models: {best_f:.4f}]")
 
final4 = best_w4[0] * test_svm + best_w4[1] * test_big + best_w4[2] * test_go + best_w4[3] * test_llm
sub4 = pd.DataFrame({"id": test_df["post_id"], "label": np.array(classes)[final4.argmax(1)]})
sub4 = sample[["post_id"]].rename(columns={"post_id": "id"}).merge(sub4, on="id", how="left")
assert sub4["label"].notna().all() and len(sub4) == len(sample)
sub4.to_csv("submission_llm.csv", index=False)
print(sub4["label"].value_counts())
 

Gemini zero-shot macro F1 on train: 0.5587
weights (SVM, roberta-large, go-roberta, Gemini) = (np.float64(0.65), np.float64(0.15), np.float64(0.1), np.float64(0.1)) -> OOF macro F1 0.6589   [3 models: 0.6554]
label
joy         371
anger       342
sadness     257
surprise    222
disgust     136
fear        112
Name: count, dtype: int64
